## Colab 준비
Colab에서 열었다면 **아래 칸을 맨 먼저 실행**하세요. 도우미 파일과 한글 글꼴을 받아요 (10~20초).

- 더 빠르게: 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**
- 고친 내용 저장: **파일 → Drive에 사본 저장**
- 맥 실습실에서 열었다면 이 칸은 아무 일도 안 해요.

In [ ]:
import os, sys, urllib.request
if "google.colab" in sys.modules:
    BASE = "https://raw.githubusercontent.com/sangwon9401/silicon-descent/main/colab/"
    for f in ['common.py', 'solutions/__init__.py', 'solutions/sol00.py', 'solutions/sol01.py', 'solutions/sol02.py', 'solutions/sol04.py']:
        os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
        urllib.request.urlretrieve(BASE + f, f)
    os.system("apt-get -qq install -y fonts-nanum > /dev/null 2>&1")
    print("Colab 준비 완료 ✅")
else:
    print("Colab이 아니에요. 그냥 넘어가요.")

# 01. Transformer 직접 짜보기

**목표**: SmolLM2-135M 모델의 계산을 처음부터 직접 짜서, HuggingFace가 만든 것과 **완전히 같은 답**을 내는 것.

사이트의 1단계 첫 강의("Transformer 구조를 손으로 다시 짜보기")를 먼저 보고 오면 훨씬 쉬워요.

> **`model`, `tok`, `W`, `cfg`나 PyTorch 함수가 낯설면 `00_python_pytorch_basics` 실습을 먼저 하세요.** HuggingFace 모델·토크나이저, common.py 도우미, 그리고 여기서 쓰는 `x[..., :half]`, `x @ w.T`, `.view`, `einsum`, `masked_fill`을 하나씩 연습해요. 30분~1시간이면 끝나요.

## 진행 방법
1. 셀을 위에서부터 하나씩 **Shift + Enter**로 실행해요.
2. `...` 이 있는 곳이 **내가 채울 부분**이에요. 설명과 힌트를 읽고 채워 보세요.
3. 바로 아래 **채점 셀**을 실행하면 ✅ 또는 ❌ 가 나와요.
4. 정말 막히면, 함수 아래에 있는 `# from solutions...` 줄의 맨 앞 `#`을 지우고 실행하면 정답 코드를 불러와요. 정답 코드는 `solutions/sol01.py`에서 읽어 볼 수 있어요.

틀려도 괜찮아요. ❌ 메시지가 어디가 틀렸는지 알려 줘요.

In [ ]:
import torch
import torch.nn.functional as F
from common import *

torch.manual_seed(0)
device = pick_device()
print("사용할 장치:", device)

## 0. 모델 불러오기

HuggingFace에서 SmolLM2-135M을 내려받아요. 처음 한 번은 약 270MB를 받아서 1~2분 걸릴 수 있어요.
이 모델은 **정답지** 역할이에요. 우리가 짠 코드의 결과를 이 모델과 비교할 거예요.

In [ ]:
model, tok = load_model(device)
cfg = model_cfg(model)
W = extract_weights(model)
print("설정:", cfg)
print("파라미터 수:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

### 0-1. 토큰 보기
이 모델은 영어로 학습돼서 영어 문장을 써요. 문장이 숫자(토큰 ID)로 바뀌는 걸 확인해 보세요.

In [ ]:
prompt = "The capital of France is"
ids = tok(prompt, return_tensors="pt").input_ids[0].to(device)
print("토큰 ID:", ids.tolist())
print("조각들 :", [tok.decode([i]) for i in ids])

### 0-2. 가중치 구경하기
`W`에는 모델의 모든 가중치가 들어 있어요. 블록 하나에 어떤 행렬이 있고 모양이 어떤지 보세요.
행렬 모양은 `[출력 크기, 입력 크기]`예요. 그래서 계산할 때는 `x @ w.T` 처럼 뒤집어서 곱해요.

In [ ]:
print("임베딩 표:", tuple(W["embed"].shape))
for name, w in W["layers"][0].items():
    print(f"블록0 {name:7s}", tuple(w.shape))

## 1. RMSNorm

벡터의 크기를 고르게 맞추는 정규화예요. 마지막 차원(576칸)에 대해:

$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\text{mean}(x^2) + \epsilon}} \times w$$

**힌트**
- 제곱: `x.pow(2)`, 마지막 차원 평균: `.mean(dim=-1, keepdim=True)`
- `keepdim=True`를 해야 `[T, 1]` 모양이 되어 `[T, 576]`과 나눌 수 있어요.
- 1/√ 은 `torch.rsqrt(...)`

In [ ]:
def rms_norm(x, weight, eps):
    ...  # 여기를 채우세요


# 막히면 아래 줄 맨 앞의 '#'을 지우고 실행하세요.
# from solutions.sol01 import rms_norm

In [ ]:
# 채점: 내 RMSNorm과 HuggingFace의 RMSNorm을 같은 입력에 돌려 비교해요.
x = torch.randn(5, 576, device=device)
run_check("RMSNorm", lambda: check_close("RMSNorm", rms_norm(x, W["layers"][0]["ln1"], cfg["eps"]),
                                         model.model.layers[0].input_layernorm(x)))

## 2. RoPE (위치 정보 넣기)

RoPE는 Q, K 벡터를 위치에 따라 **회전**시켜요. 회전 각도(cos, sin)를 만드는 함수는 미리 짜 뒀어요.
여러분은 두 함수를 채우면 돼요.

**rotate_half**: 마지막 차원을 반으로 나눠 앞쪽 `x1`, 뒤쪽 `x2`라 할 때 `[-x2, x1]`을 만들어요.
- 반 크기: `half = x.shape[-1] // 2`
- 자르기: `x[..., :half]`, `x[..., half:]`
- 붙이기: `torch.cat([a, b], dim=-1)`

**apply_rope**: `x * cos + rotate_half(x) * sin`
- `x`는 `[T, 헤드 수, 64]`, `cos`와 `sin`은 `[T, 64]`라서 가운데에 차원을 하나 끼워야 해요: `cos[:, None, :]`

In [ ]:
def rope_cos_sin(positions, head_dim, theta, device):
    """(미리 짜 둔 함수) 위치마다 회전 각도의 cos, sin을 만들어요. 결과 모양: [T, head_dim]"""
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    freqs = positions.float()[:, None] * inv_freq[None, :]
    emb = torch.cat([freqs, freqs], dim=-1)
    return emb.cos(), emb.sin()


def rotate_half(x):
    ...  # 여기를 채우세요


def apply_rope(x, cos, sin):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol01 import rotate_half, apply_rope

In [ ]:
from solutions import sol01 as ans

xr = torch.randn(7, 9, 64, device=device)
c, s = rope_cos_sin(torch.arange(7, device=device), 64, cfg["theta"], device)
run_check("rotate_half", lambda: check_close("rotate_half", rotate_half(xr), ans.rotate_half(xr)))
run_check("apply_rope", lambda: check_close("apply_rope", apply_rope(xr, c, s), ans.apply_rope(xr, c, s)))

## 3. 어텐션 (GQA + causal mask)

이번 실습의 핵심이에요. 입력 모양은 이래요.
- `q`: `[T, 9, 64]` (이번에 계산할 토큰 T개, Q 헤드 9개)
- `k`, `v`: `[S, 3, 64]` (지금까지의 전체 토큰 S개, KV 헤드 3개). 지금은 S = T예요. 다음 실습(KV 캐시)에서 S > T가 돼요.

**순서**
1. **GQA**: KV 헤드 3개를 Q 헤드 9개에 맞춰 늘려요. 헤드 하나를 3번씩 복사: `k.repeat_interleave(3, dim=1)` (3 = 9 // 3)
2. **점수**: 헤드마다 Q·K를 계산하고 √64로 나눠요. `torch.einsum("tnd,snd->nts", q, k)`를 쓰면 `[9, T, S]`가 나와요.
3. **미래 가리기**: 이번 토큰들의 실제 위치는 `S-T, ..., S-1`이에요. 내 위치보다 뒤에 있는 K는 점수를 `-inf`로 만들어요.
   - `qpos = torch.arange(S - T, S)[:, None]`, `kpos = torch.arange(S)[None, :]`
   - `scores.masked_fill(kpos > qpos, float("-inf"))`  (장치를 맞추려면 `device=q.device`도 넣어요)
4. **softmax**: `scores.softmax(dim=-1)`
5. **V 섞기**: `torch.einsum("nts,snd->tnd", probs, v)` → `[T, 9, 64]`

In [ ]:
def attention(q, k, v):
    T, nq, hd = q.shape
    S, nkv, _ = k.shape
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol01 import attention

In [ ]:
q_ = torch.randn(6, 9, 64, device=device)
k_ = torch.randn(6, 3, 64, device=device)
v_ = torch.randn(6, 3, 64, device=device)
run_check("어텐션 (S = T)", lambda: check_close("어텐션 (S = T)", attention(q_, k_, v_), ans.attention(q_, k_, v_)))
# KV 캐시 상황 흉내: 토큰 2개만 새로 계산하고, K/V는 전체 6개
run_check("어텐션 (S > T)", lambda: check_close("어텐션 (S > T)", attention(q_[-2:], k_, v_), ans.attention(q_[-2:], k_, v_)))

## 4. MLP (SwiGLU)

$$\text{MLP}(x) = \big(\text{SiLU}(x W_{gate}^T) \odot (x W_{up}^T)\big) W_{down}^T$$

- SiLU: `F.silu(...)`
- ⊙ 는 같은 자리끼리 곱하기, 그냥 `*` 예요.

In [ ]:
def mlp(x, w_gate, w_up, w_down):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol01 import mlp

In [ ]:
lw0 = W["layers"][0]
xm = torch.randn(4, 576, device=device)
run_check("MLP", lambda: check_close("MLP", mlp(xm, lw0["w_gate"], lw0["w_up"], lw0["w_down"]),
                                    model.model.layers[0].mlp(xm), atol=1e-4))

## 5. 디코더 블록 조립

지금까지 만든 부품으로 블록 하나를 만들어요. 사이트 그림의 순서 그대로예요.

```
h = RMSNorm(x, ln1)
q = h @ wq.T  → [T, 9, 64] 로 모양 바꾸기  (.view(T, cfg["nq"], cfg["hd"]))
k = h @ wk.T  → [T, 3, 64]
v = h @ wv.T  → [T, 3, 64]
q, k 에 RoPE 적용   (v에는 안 해요!)
a = attention(q, k, v) 를 [T, 576]으로 펴서 (.reshape(T, -1)) @ wo.T
x = x + a                                   ← 잔차
x = x + mlp(RMSNorm(x, ln2), ...)           ← 잔차
```

In [ ]:
def decoder_layer(x, lw, cfg, cos, sin):
    T = x.shape[0]
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol01 import decoder_layer

In [ ]:
xd = torch.randn(5, 576, device=device)
cd, sd = rope_cos_sin(torch.arange(5, device=device), cfg["hd"], cfg["theta"], device)
run_check("디코더 블록", lambda: check_close("디코더 블록", decoder_layer(xd, lw0, cfg, cd, sd),
                                          ans.decoder_layer(xd, lw0, cfg, cd, sd), atol=1e-4))

## 6. 전체 forward

```
x = 임베딩 표에서 토큰 ID 줄 꺼내기        W["embed"][tokens]
cos, sin = rope_cos_sin(0..T-1 위치)
블록 30개를 차례로 통과
x = RMSNorm(x, W["norm"])
logits = x @ W["lm_head"].T                 → [T, 49152]
```

이 셀의 채점이 **진짜 시험**이에요. HuggingFace 모델의 출력과 비교해요.

In [ ]:
def forward(tokens, W, cfg):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol01 import forward

In [ ]:
with torch.no_grad():
    ref_logits = model(ids[None]).logits[0]
    run_check("전체 forward vs HuggingFace",
              lambda: check_close("전체 forward vs HuggingFace", forward(ids, W, cfg), ref_logits, atol=1e-3, rtol=1e-3))

## 7. 문장 만들기 (greedy)

forward를 반복 호출해서 토큰을 하나씩 붙여요.
```
n_new번 반복:
    logits = forward(지금까지 토큰)
    다음 토큰 = 마지막 위치 logits 중 가장 큰 것의 번호   logits[-1].argmax()
    토큰 뒤에 붙이기                                       torch.cat([tokens, nxt[None]])
```
지금은 매번 처음부터 전부 다시 계산해요. 이게 얼마나 낭비인지는 다음 실습에서 재 봐요.

In [ ]:
def greedy_generate(tokens, W, cfg, n_new):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol01 import greedy_generate

In [ ]:
with torch.no_grad():
    hf_out = model.generate(ids[None], max_new_tokens=15, do_sample=False)[0]
    mine = greedy_generate(ids, W, cfg, 15)
if run_check("문장 생성 vs HuggingFace", lambda: check_equal("문장 생성 vs HuggingFace", None if mine is None else mine.tolist(), hf_out.tolist())):
    print("\n내 모델이 쓴 문장:", tok.decode(mine))

## 끝! 🎉

모든 채점이 ✅ 라면, 여러분은 LLM 하나를 처음부터 직접 돌린 거예요. vLLM이나 SGLang도 결국 이 계산을 **더 빠르게, 더 많이 동시에** 하는 프로그램이에요.

**스스로 확인해 보기**
- 위 문장 생성 셀에서 `15`를 `50`으로 바꾸면 시간이 얼마나 늘어나나요? 왜 그럴까요?
- `prompt`를 다른 영어 문장으로 바꿔 보세요.

다음: **02_kv_cache.ipynb** 에서 이 낭비를 없애요.